In [2]:
import requests
import os
import json
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

def create_embedding(text_list):
    #https://github.com/ollama/ollama/blob/main/docs/api.md#generatre-embeddings
    r=requests.post("http://localhost:11434/api/embed",json={
        "model" : "bge-m3",
        "input" : text_list
    })
    embedding= r.json()["embeddings"]
    return embedding

jsons=os.listdir("/content/drive/MyDrive/RAG-project/jsons")     #Listed all the jsons
chunk_id=0

my_dict=[]
for json_file in jsons:
    with open(f"/content/drive/MyDrive/RAG-project/jsons/{json_file}") as f:
        content=json.load(f)
    print(f"Creating embeddings for {json_file}")
    embeddings=create_embedding([c["text"] for c in content["chunks"]])
    for i,chunk in enumerate(content["chunks"]):
        chunk["chunk_id"]=chunk_id
        chunk["embedding"]=embeddings[i]
        chunk_id+=1
        my_dict.append(chunk)
    #break
    
#print(my_dict)

df=pd.DataFrame.from_records(my_dict)
print(df)


# text_list=create_embedding(["I am Adil Ahmed","I am a good boy"])
# print(text_list)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating embeddings for 10_Video, Audio & Media in HTML.mp3.json


TypeError: list indices must be integers or slices, not str

In [1]:
import os
import json
import pandas as pd

json_dir = "/content/drive/MyDrive/RAG-project/jsons"
jsons = sorted(
    f for f in os.listdir(json_dir)
    if f.endswith(".json")
)

chunk_id = 0
my_dict = []

for json_file in jsons:
    file_path = os.path.join(json_dir, json_file)

    with open(file_path, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    print(f"Creating embeddings for {json_file}")

    # Your JSON contains a list of chunks directly
    if not isinstance(chunks, list):
        print(f"Skipping unexpected JSON structure: {json_file}")
        continue

    # Generate embeddings in batches to avoid oversized requests
    batch_size = 8

    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        texts = [c["text"] for c in batch]

        embeddings = create_embedding(texts)

        for chunk, embedding in zip(batch, embeddings):
            chunk["chunk_id"] = chunk_id
            chunk["embedding"] = embedding
            chunk["source_file"] = json_file

            my_dict.append(chunk)
            chunk_id += 1

df = pd.DataFrame.from_records(my_dict)
print(df.head())
print("Total embedded chunks:", len(df))


Creating embeddings for 0_sample.mp3.json
Skipping unexpected JSON structure: 0_sample.mp3.json
Creating embeddings for 10_Video, Audio & Media in HTML.mp3.json


NameError: name 'create_embedding' is not defined